# Global Co2 Emissions Responsibility : Total vs Per-Capita vs Historical Accountability Dashboard

## Objectives

The objective of this notebook is to perform the ETL (Extract, Transform, Load) process for the CO₂ emissions dataset.

This notebook aims to:
Ensure that the data was clean, consistent, and suitable for exploratory analysis, hypothesis testing, and predictive modelling.
Extract the raw CO2 dataset obtained from Our World in Data and stored locally in DataSet/Raw/
Clean and prepare the data for analysis
Filter the dataset to focus on relevant years (from 1990 onward for trend analysis)
Create derived metrics such as cumulative emissions and growth rates
Prepare structured datasets for use in the BI dashboard
This ensures the data is accurate, consistent, and ready for analysis and visualisation. 

## Inputs
The notebook requires the following input : 
DataSet/Raw/Co2_emissions.csv
Key columns used are:
country
year
co2
co2_per_capita
population
gdp
The dataset was sourced from Our World in Data and contains aggregated country-level CO₂ emissions statistics.


## Outputs
- Cleaned DataSet: DataSet/Cleaned/co2_emissions_clean.csv
-Exported/ Dashboard ready datasets

## Additional Comments
The year 1990 was selected as the baseline for trend analysis because it is widely used in international climate agreements and provides consistent reporting coverage.
Only aggregated national data is used; no personal data is included.
All data transformations are documented in this notebook to ensure transparency and reproducibility.
Version control is used to track changes and maintain governance best practices.



Change working directory
Notebooks are stored in the jupyter_notebooks/ subfolder. This cell checks the current working directory and navigates to the project root (Global-Co2-Emissions-Responsibility/) if needed, ensuring relative paths to data/ work correctly.

In [1]:
import os
current_dir = os.getcwd()
current_dir

'\\\\talktalk\\redirectedfolders\\F.Afolabi\\Documents\\Global Co2 Emissions project 3\\Global-Co2-Emissions-Responsibility\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(r"\\talktalk\redirectedfolders\F.Afolabi\Documents\Global Co2 Emissions project 3\Global-Co2-Emissions-Responsibility")
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'\\\\talktalk\\redirectedfolders\\F.Afolabi\\Documents\\Global Co2 Emissions project 3\\Global-Co2-Emissions-Responsibility'

# Section 1

This section sets the project root as the working directory and loads the raw CSV for inspection.



Extraction Phase

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



# Data Preparation

In [5]:
# Load the dataset
df_co2_emissions= pd.read_csv("DataSet/Raw/Co2_emissions.csv")
df_co2_emissions.head()

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## The dataset is loaded from the raw CSV file into a pandas DataFrame for inspection and cleaning.
What this code does: Loads the raw dataset and previewed the first 5 rows.

Key observations from the raw extract:

Many early-year records (e.g., 1750–1800) have missing values for GDP and CO₂.

Columns include population, GDP, CO₂ metrics, energy-related emissions, and growth indicators.

In [6]:
# Understanding the dataset, identify relevant columns and preparing for cleaning.
df_co2_emissions.shape
df_co2_emissions.columns
df_co2_emissions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50411 entries, 0 to 50410
Data columns (total 79 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    50411 non-null  object 
 1   year                                       50411 non-null  int64  
 2   iso_code                                   42480 non-null  object 
 3   population                                 41167 non-null  float64
 4   gdp                                        15251 non-null  float64
 5   cement_co2                                 29173 non-null  float64
 6   cement_co2_per_capita                      25648 non-null  float64
 7   co2                                        29384 non-null  float64
 8   co2_growth_abs                             27216 non-null  float64
 9   co2_growth_prct                            26239 non-null  float64
 10  co2_including_luc     

## Before cleaning, the dataset is profiled to understand completeness and relevance.
Initial Inspection:  df.info() was used to identify the schema.This helped to identfy the Data types, missing values, Dataset size and key variables.
The dataset contains 50,411 rows and 79 columns.
Several columns such as CO2 per capita, GDP and Population have extensive missing values.

Only a subset of columns is relevant for modelling and analysis.

GDP and population coverage varies significantly across years.

## Transformation Phase

In [7]:
# Keeping only the relevant columns for our analysis.
keeps_columns = ["country", "iso_code", "year", "co2", "co2_per_capita", "gdp", "population", "cumulative_co2",]



## To simplify the dataset and focus on modelling, only essential fields are retained.

The step removes unnecessary variables from the orinal dataset and keeps only the field relevant for 
- Trend Analysis
- Per-Capita comparison
- Historical resposibility
- Predictive Modelling 

These columns matters because: 
country / iso_code / year will be  identifiers for grouping and filtering

co2 / co2_per_capita will be target variables for modelling

gdp / population  will be  key predictors for multi‑factor regression

cumulative_co2 will be  useful for historical context and trend analysis

In [8]:
# verify the columns we are keeping
df_co2_emissions = df_co2_emissions[keeps_columns]
df_co2_emissions.head()

,country,iso_code,year,co2,co2_per_capita,gdp,population,cumulative_co2
0,Afghanistan,AFG,1750,NaN,NaN,NaN,2802560.0,NaN
1,Afghanistan,AFG,1751,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1752,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1753,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1754,NaN,NaN,NaN,NaN,NaN


## Filtering the Dataset
This step filters teh dataset to include only the selected columns(all the columns we are keeping)
It ensures that subsequest analysis is focused and structured around the project objectives.
While this code: df_co2_emissions.head(): displays the first five rows of the cleaned dataset to verify that:

The correct columns were retained, the data loaded correctly and no structural errors occurred.

In [9]:
df_co2_emissions.shape

(50411, 8)

## Checking Dataset Dimensions
This confirm the dataset contain 50,411 rows and 8 columns.
Understanding the dataset size helps evaluate completeness and coverage

In [11]:
# Checking for missing values
df_co2_emissions.isnull().sum()

country               0
iso_code           7931
year                  0
co2               21027
co2_per_capita    23902
gdp               35160
population         9244
cumulative_co2    22848
dtype: int64

## Data Structure and Missing Value Assessment
This identifies missing values in each column
Initial inspection revealed significant missing values in key analytical fields:
The CO₂ emissions column contains only 29,384 non-null record, 21,027 null values
Per-capita emissions and cumulative emissions also contain 23,902 and 22,848 missing values respectively.
GDP data shows substantial missingness and will be treated as an optional analytical variable.

This step is critical because missing values influence: Hypothesis testing, Regression modelling and predictive accuracy and ensure statistical reliability.

In [12]:
# Describe the dataset to understand the distribution of values and identify any anomalies or outliers.
df_co2_emissions.describe()

,year,co2,co2_per_capita,gdp,population,cumulative_co2
count,50411.000000,29384.000000,26509.000000,1.525100e+04,4.116700e+04,2.756300e+04
mean,1920.349249,420.227031,3.821372,3.300794e+11,6.017453e+07,1.249223e+04
std,65.859123,1972.092036,14.312865,3.087720e+12,3.308433e+08,7.312143e+04
min,1750.000000,0.000000,0.000000,4.998000e+07,2.150000e+02,0.000000e+00
25%,1875.000000,0.381056,0.171333,7.874038e+09,3.272140e+05,4.238027e+00
50%,1925.000000,5.080755,1.023368,2.743861e+10,2.291594e+06,8.046645e+01
75%,1975.000000,53.656342,4.327494,1.210000e+11,9.986553e+06,1.163685e+03
max,2024.000000,38598.578130,782.743408,1.300000e+14,8.161973e+09,1.849124e+06


## Statistical Summary of Key Variables
The dataset spans from 1750 to 2024.
Significant missing values were identified:
Only 29,384 records contain total CO₂ emissions.
Per-capita and cumulative emissions also show substantial null values 
GDP data is heavily incomplete and will be treated as an optional analytical variable.
Based on this assessment, rows lacking emissions data will be removed, and the dataset will be filtered from 1990 onward to ensure consistency and policy relevance.

In [13]:
# Drop rows with missing values
df_co2_emissions = df_co2_emissions.dropna(subset=["co2",])

## Rows with missing values were removed only when necessary for specific analyses.

In [14]:
# Filtering the dataset to include only records from 1990 onwards, as this period is more relevant for our analysis of global CO2 emissions.
df_co2_emissions = df_co2_emissions[df_co2_emissions["year"] >= 1990]   


## Filtering Data to Relevant Time period (1990 Onwards)
This step fiilters the dataset to include only records from 1990 onwards.

The decision to restrict analysis to post-1990 data reflects a methodological choice to balance historical accountability with contemporary policy relevance. While earlier data exists, focusing on the modern emissions era provides more actionable insight for current climate responsibility discussions.


In [15]:
# Checking the information of the dataset after cleaning
df_co2_emissions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8617 entries, 240 to 50410
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         8617 non-null   object 
 1   iso_code        7508 non-null   object 
 2   year            8617 non-null   int64  
 3   co2             8617 non-null   float64
 4   co2_per_capita  8068 non-null   float64
 5   gdp             5420 non-null   float64
 6   population      7893 non-null   float64
 7   cumulative_co2  8204 non-null   float64
dtypes: float64(5), int64(1), object(2)
memory usage: 605.9+ KB


## 

In [16]:
#Checking for missing values after cleaning the dataset  

df_co2_emissions.isnull().sum()

country              0
iso_code          1109
year                 0
co2                  0
co2_per_capita     549
gdp               3197
population         724
cumulative_co2     413
dtype: int64


## This step re-evaluates missing values after filtering.
Key observation: 
GDP Still contains 3,197 missing values
CO2 per capita has 549  incomplete records
population has 724 moderate missing values
The core variables; country, year and total CO2 are complete.




In [17]:
# checking the ISO codes in the dataset to indentify the unique countries represented in and the missing values in the iso_code column.
df_co2_emissions["iso_code"].unique()

array(['AFG', nan, 'ALB', 'DZA', 'AND', 'AGO', 'AIA', 'ATA', 'ATG', 'ARG',
       'ARM', 'ABW', 'AUS', 'AUT', 'AZE', 'BHS', 'BHR', 'BGD', 'BRB',
       'BLR', 'BEL', 'BLZ', 'BEN', 'BMU', 'BTN', 'BOL', 'BES', 'BIH',
       'BWA', 'BRA', 'VGB', 'BRN', 'BGR', 'BFA', 'BDI', 'KHM', 'CMR',
       'CAN', 'CPV', 'CAF', 'TCD', 'CHL', 'CHN', 'CXR', 'COL', 'COM',
       'COG', 'COK', 'CRI', 'CIV', 'HRV', 'CUB', 'CUW', 'CYP', 'CZE',
       'COD', 'DNK', 'DJI', 'DMA', 'DOM', 'TLS', 'ECU', 'EGY', 'SLV',
       'GNQ', 'ERI', 'EST', 'SWZ', 'ETH', 'FRO', 'FJI', 'FIN', 'FRA',
       'PYF', 'GAB', 'GMB', 'GEO', 'DEU', 'GHA', 'GRC', 'GRL', 'GRD',
       'GTM', 'GIN', 'GNB', 'GUY', 'HTI', 'HND', 'HKG', 'HUN', 'ISL',
       'IND', 'IDN', 'IRN', 'IRQ', 'IRL', 'ISR', 'ITA', 'JAM', 'JPN',
       'JOR', 'KAZ', 'KEN', 'KIR', 'KWT', 'KGZ', 'LAO', 'LVA', 'LBN',
       'LSO', 'LBR', 'LBY', 'LIE', 'LTU', 'LUX', 'MAC', 'MDG', 'MWI',
       'MYS', 'MDV', 'MLI', 'MLT', 'MHL', 'MRT', 'MUS', 'MEX', 'FSM',
       'MDA', '

In [18]:
# Removing rows with missing Nan ISO codes, as these are essential for our analysis and visualization.
df_co2_emissions = df_co2_emissions[df_co2_emissions["iso_code"].notna()]   

In [19]:
# Verifying the total number of unique countries in the dataset after cleaning and the df shape
df_co2_emissions.isna().sum()


country              0
iso_code             0
year                 0
co2                  0
co2_per_capita      66
gdp               2099
population          66
cumulative_co2       0
dtype: int64

In [20]:
df_co2_emissions.shape
df_co2_emissions["country"].nunique()

215

## Unique Countries 
This step identifies unique country codes, missing ISO codes(NaN values). ISO codes are important for country-level grouping, Dashboard filtering and prevent duplicate country entries.
Following ISO code filtering, the dataset contains 215 unique countries and territories. This aligns with global emissions reporting standards, which include sovereign nations as well as dependent territories with independently recorded emissions data.

---

## Clean Phase

In [21]:
# save the cleaned dataset to a new csv file for future use
df_co2_emissions_clean = df_co2_emissions.copy()   
df_co2_emissions_clean.to_csv("DataSet/Cleaned/Co2_emissions_cleaned.csv", index=False) 



## Clean dataset saved
 A cleaned version of the dataset was stored in a new DataFrame named df_co2_emissions_clean. This separation ensures that the original extracted dataset remains unchanged while providing a dedicated structure for analysis and dashboard preparation.

# Conclusion
he Extract, Transform, and Load (ETL) process was conducted to prepare the Global CO₂ Emissions dataset for structured analysis, hypothesis testing, and predictive modelling.

During the Extraction phase, the dataset was sourced from Our World in Data (OWID) and loaded into a Pandas DataFrame for inspection and validation.

In the Transformation phase, the following steps were applied:

Selected only relevant variables required for analysis.filtered the dataset to include records from 1990 onwards to align with international climate policy baselines, verified data types and structural integrity.

Assessed and documented missing values across key variables, Identified incomplete GDP and per-capita reporting.

Reviewed ISO codes to ensure country-level consistency.

In the Load phase, cleaned datasets were saved into structured project directories for reuse in downstream analysis and dashboard development.

This process ensured that the dataset was:Structurally consistent, relevant to the research objectives,  Free from unnecessary variables and suitable for statistical modelling.


 

